In [1]:
%pip install -U transformers accelerate peft trl datasets sentencepiece bitsandbytes sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [2]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

from trl import SFTTrainer
from peft import prepare_model_for_kbit_training

In [3]:
#here we are going to finetune this model the english to hindi translation
dataset = load_dataset("ai4bharat/samanantar", "hi")

README.md:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

hi/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  240MB            

hi/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

hi/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  240MB            

hi/train-00001-of-00008.parquet: downloading bytes:           |  0.00B            

...

hi/train-00006-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  239MB            

hi/train-00006-of-00008.parquet: downloading bytes:           |  0.00B            

hi/train-00007-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  240MB            

hi/train-00007-of-00008.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10125706 [00:00<?, ? examples/s]

In [4]:
dataset  #it has 10125706 rows

DatasetDict({
    train: Dataset({
        features: ['idx', 'src', 'tgt'],
        num_rows: 10125706
    })
})

In [5]:
dataset['train'][0]

{'idx': 0,
 'src': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles",
 'tgt': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।'}

In [6]:
dataset["train"] = dataset["train"].select(range(1000))  # use only 1k samples

In [7]:
dataset['train'][0]

{'idx': 0,
 'src': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles",
 'tgt': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।'}

In [8]:
#now lets load the tokenizer of llama-3.2-1b-instruct
model_name = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [9]:
tokenizer.pad_token = tokenizer.eos_token #here we have added <eos> as a padding tokens for handling the differnt length of inputs in batch

In [10]:
#now lets format this dataset for the finetuning
#format this dataset
def formatting_func(example):

    text = f"""### Instruction:
{example['src']}



### Response:
{example['tgt']}"""

    return {"text": text}


In [11]:
dataset=dataset.map(formatting_func)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [12]:
dataset['train']['text'][0]

"### Instruction:\nHowever, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles\n\n\n\n### Response:\nआस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।"

In [13]:
#lets see the tokenization
token_ids=tokenizer(dataset['train']['text'][0])['input_ids']
token_ids

[128000,
 14711,
 30151,
 512,
 11458,
 11,
 16056,
 288,
 11,
 889,
 574,
 70220,
 8494,
 596,
 7043,
 21296,
 3258,
 11,
 1436,
 1193,
 733,
 439,
 3117,
 439,
 279,
 8502,
 12085,
 82,
 1405,
 814,
 5675,
 311,
 31930,
 455,
 67631,
 323,
 14521,
 645,
 1038,
 14711,
 6075,
 512,
 102393,
 79468,
 100431,
 86133,
 101385,
 100322,
 24810,
 48909,
 35470,
 84736,
 100391,
 85410,
 101276,
 92911,
 44747,
 48909,
 35470,
 69258,
 101029,
 100277,
 114868,
 44747,
 101403,
 100306,
 35470,
 100287,
 100391,
 35470,
 84736,
 101755,
 92317,
 100322,
 100497,
 44747,
 92317,
 100271,
 48909,
 100460,
 100273,
 105461,
 100924,
 106357,
 92911,
 102007,
 85410,
 44747,
 101372,
 103515,
 100411,
 101026,
 35470,
 48909,
 100305,
 100311,
 65804,
 39951,
 100504,
 100291,
 103008,
 92317,
 100271,
 100918,
 101358,
 100271,
 100348,
 101843,
 80338,
 39951,
 100358,
 100282,
 101758,
 100782,
 100282,
 35470,
 104579,
 100537,
 24810,
 100518,
 100781]

In [14]:
#lets see the tokens again
tokenizer.decode(token_ids)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


"<|begin_of_text|>### Instruction:\nHowever, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles\n\n\n\n### Response:\nआस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।"

In [15]:
#lets make the configuration for the 4 bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # ← change to bfloat16
    bnb_4bit_use_double_quant=True,
)

In [16]:
#now, we are loading 4-bit quantized llama-3.2-1b-instruct llm for finetuning
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 #device_map='auto' Automatically place the model on the available GPU
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [17]:
#lets check our LLM
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [18]:
#lets count the trainable parameters from this LLM
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 749,275,136
Trainable Parameters: 262,735,872


In [19]:
#now lets freeze the trainable parameters of the LLM
model = prepare_model_for_kbit_training(model)

In [20]:
#lets count the trainable parameters from this LLM
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 749,275,136
Trainable Parameters: 0


In [21]:
#here we are finetuning using LoRA technique
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

In [22]:
#lets attach the LoRA to llama-3.2-1b-instruct LLM
model = get_peft_model(model, lora_config)

In [23]:
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         param.data = param.data.to(torch.float16)
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.bfloat16)

In [24]:
model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [25]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [26]:
#now lets configure the arguments for retraining
training_args = TrainingArguments(
    output_dir="content/drive/llama_lora",

    num_train_epochs=3,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    fp16=True,
    bf16=False,  #before fp16=True ad bf16=False

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [27]:
#lets create the trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"], #dataset for finetuning
    processing_class=tokenizer,
    args=training_args,
    formatting_func=lambda example: example["text"], #input for the LLM
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [28]:
#lets do fine-tuning
# trainer.train()
# trainer.train(resume_from_checkpoint=True)


In [29]:
#now we have trained the llm successfully lets load the finetuned LLM and then test it

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_name = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model_for_lora=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
# Path where you saved your model in Google Drive
LORA_PATH = "/content/drive/MyDrive/llama_lora/checkpoint-375"

fine_tuned_model = PeftModel.from_pretrained(model_for_lora, LORA_PATH)
fine_tuned_model.eval()  # set to inference mode
print("✅ LoRA adapters loaded!")

✅ LoRA adapters loaded!


In [32]:
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)
print("✅ Tokenizer loaded!")

✅ Tokenizer loaded!


In [33]:
# NOTE:
# The model was fine-tuned using the "### Instruction" and "### Response"
# prompt format. For consistency, the same template is used during inference.
# "### Input" would also be a valid choice, but changing the prompt format
# after training is not recommended.

In [34]:
#now lets see the difference between our finetuned LLM with the original LLM


# This is the function for inference using our finetuned LLM
def translate(english_text):
    prompt = (
        "### Instuction:\n"
        f"{english_text}\n\n\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)

    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Test it!
test_sentences = [
    "I love machine learning",
    "The weather is nice today.",
    "I love learning artificial intelligence.",
]

for sentence in test_sentences:
    print(f"EN: {sentence}")
    print(f"HI: {translate(sentence)}")
    print()


#In the response we can easily see that the finetuned llm is translating the text without any extra Instruction

EN: I love machine learning
HI: मैं मशीन लर्निंग को बहुत पसंद करता हूँ।

EN: The weather is nice today.
HI: सunny weather today.

EN: I love learning artificial intelligence.
HI: मैं AI को सीखना पसंद करता हूं।



In [35]:
#now lets see the difference with the original LLM along with the Instruction
def translate_with_base_llm(english_text):
    prompt = (
        "### Instruction: Translate the following text from English to Hindi\n"
        f"### Input: {english_text}\n\n\n\n"
        "### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Test it!
test_sentences = [
    "I love machine learing",
    "The weather is nice today.",
    "I love learning artificial intelligence.",
]

for sentence in test_sentences:
    print(f"EN: {sentence}")
    print(f"HI: {translate(sentence)}")
    print()

EN: I love machine learing
HI: मैं मशीन लर्निंग को बहुत पसंद करता हूँ।

EN: The weather is nice today.
HI: सunny weather today.

EN: I love learning artificial intelligence.
HI: मैं AI को सीखना पसंद करता हूं।



In [36]:
#now lets evaluate the quality of our translation LLM

In [37]:
#here we will use the BLEU(Bilingual Evaluation Understudy) Precision Score for the evaluation

from datasets import load_dataset
#here we are going to finetune this model the english to hindi translation

dataset = load_dataset("ai4bharat/samanantar", "hi")

In [38]:
dataset

DatasetDict({
    train: Dataset({
        features: ['idx', 'src', 'tgt'],
        num_rows: 10125706
    })
})

In [39]:
#NOTE: Here for the finetuning I have used only 1k rows so the finetuned model is not that great
#because we have limit cpu and gpu in colab with limited runtime
#we will going to use only 10 rows for the evaluation
testing_dataset=dataset['train'].select(range(1000,1010))

In [40]:
def formatting_function(example):
  input=example['src']

  prompt=f"""###Instruction: \n
  {input} \n\n

  ###Response: \n """
  return {'text': prompt}

In [41]:
testing_dataset

Dataset({
    features: ['idx', 'src', 'tgt'],
    num_rows: 10
})

In [42]:
#now lets format it
testing_dataset=testing_dataset.map(formatting_function)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [43]:
testing_dataset

Dataset({
    features: ['idx', 'src', 'tgt', 'text'],
    num_rows: 10
})

In [44]:
print(testing_dataset[0]['text'])

###Instruction: 

  US Secretary of State, Mike Pompeo 



  ###Response: 
 


In [45]:
from sacrebleu import corpus_bleu

In [50]:
import torch

def generate_responses(dataset, model, tokenizer):
    """
    Generates translations using the fine-tuned model and
    stores them in a new column called 'generated_response'.

    Args:
        dataset: HuggingFace Dataset
        model: Fine-tuned LLM
        tokenizer: Corresponding tokenizer

    Returns:
        Updated dataset with 'generated_response' column.
    """
    count=1
    generated_responses = []

    for sample in dataset:

        prompt = sample["text"]   # Uses the same template used during training

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = outputs[0][inputs["input_ids"].shape[1]:]
        print("Response: ",count,'generated')
        response = tokenizer.decode(
            generated,
            skip_special_tokens=True
        ).strip()

        generated_responses.append(response)
        count+=1

    dataset = dataset.add_column(
        "generated_response",
        generated_responses
    )

    return dataset

In [51]:
#lets generate the translation from our finetuned LLM
testing_dataset=generate_responses(testing_dataset,fine_tuned_model,tokenizer)

Response:  1 generated
Response:  2 generated
Response:  3 generated
Response:  4 generated
Response:  5 generated
Response:  6 generated
Response:  7 generated
Response:  8 generated
Response:  9 generated
Response:  10 generated


In [ ]:
testing_dataset

In [52]:
#lets see the generated vs references
for row in testing_dataset:
  print('Input Text: ',row['src'])
  print('Generated Translation: ',row['generated_response'])
  print('Reference Translation: ',row['tgt'])

Input Text:  US Secretary of State, Mike Pompeo
Generated Translation:  अमेरिकी राष्ट्रपति डॉ. माइक पोम्पो
Reference Translation:  माइक पोंपियो , अमेरिकी विदेश मंत्री&nbsp
Input Text:  Sridevi was with her husband Boney Kapoor and daughter Khushi at the time of death.
Generated Translation:  स्ट्राइडीवी के मृतक होने के बाद उनका परिवार भारत और पाकिस्तान में विरोधप्रदर्शन कर रहा है।
Reference Translation:  श्रीदेवी की मौत के वक्त उनके पति बोनी कपूर और बेटी खुशी उनके साथ थीं.
Input Text:  Referring to an official communication received by him from Union Minister of State, Tourism and Culture Prahlad Singh Patel said the project would not only attract tourists but will boost economy of the local populace.
Generated Translation:  इस बात पर उन्होंने कहा कि यह योजना संस्थानीय लोगों की आर्थिक वृद्धि को बढ़ावा देने के साथ-साथ पर्यटकों को आकर्षित करने में सफल होगी।
Reference Translation:  केंद्रीय राज्य मंत्री, पर्यटन और संस्कृति प्रहलाद सिंह पटेल से हुई आधिकारिक संवाद का हवाला देते हुए कहा कि य

In [53]:
#now lets evaluate
from sacrebleu import corpus_bleu

def calculate_bleu(dataset):
    """
    Calculates BLEU score using:
    generated_response -> Model Prediction
    tgt -> Reference Translation
    """

    predictions = dataset["generated_response"]
    references = dataset["tgt"]

    bleu = corpus_bleu(
        predictions,
        [references]
    )

    print(f"BLEU Score: {bleu.score:.2f}")

    return bleu.score

In [54]:
bleu_score=calculate_bleu(testing_dataset)

BLEU Score: 8.17
